[![Labellerr](https://storage.googleapis.com/labellerr-cdn/%200%20Labellerr%20template/notebook.webp)](https://www.labellerr.com)

# **AI Powered Real-Time Alphabet Recognition**

---

[![labellerr](https://img.shields.io/badge/Labellerr-BLOG-black.svg)](https://www.labellerr.com/blog/<BLOG_NAME>)
[![Youtube](https://img.shields.io/badge/Labellerr-YouTube-b31b1b.svg)](https://www.youtube.com/@Labellerr)
[![Github](https://img.shields.io/badge/Labellerr-GitHub-green.svg)](https://github.com/Labellerr/Hands-On-Learning-in-Computer-Vision)

## Project Overview  
A real-time assistive technology that translates American Sign Language (ASL) finger-spelled gestures into fluid on-screen text.

The system combines:
- Hand landmark tracking  
- Geometric feature extraction  
- Machine learning classification  
- Temporal stabilization  

to generate accurate live text synthesis from hand gestures.

---

## Pipeline Architecture  

```text
[Camera Feed]
      ↓
[MediaPipe Hand Tracking]
      ↓
[Relative Distance Vectorization]
      ↓
[Random Forest Classifier]
      ↓
[Temporal Debounce Buffer]
      ↓
[Live Text Output]

## Annotate your Custom dataset using Labellerr

 ***1. Visit the [Labellerr](https://www.labellerr.com/?utm_source=githubY&utm_medium=social&utm_campaign=github_clicks) website and click **“Sign Up”**.*** 

 ***2. After signing in, create your workspace by entering a unique name.***

 ***3. Navigate to your workspace’s API keys page (e.g., `https://<your-workspace>.labellerr.com/workspace/api-keys`) to generate your **API Key** and **API Secret**.***

 ***4. Store the credentials securely, and then use them to initialise the SDK or API client with `api_key`, `api_secret`.*** 



## Import Libraries

This section imports all the required libraries used throughout the project for computer vision, visualization, deep learning, and structured coding.


In [1]:
%pip install pandas scikit-learn joblib


[notice] A new release of pip available: 22.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Sequential ASL Data Collection Pipeline

## Overview  
This script acts as the data collection and engineering pipeline for the ASL recognition system.  
It captures normalized hand landmark data and compiles it into a structured CSV dataset for machine learning training.

---

## Core Modules

### Structural Landmark Tracking  
Uses **MediaPipe Hands** to extract 21 hand landmarks `(X, Y, Z)` with high-confidence tracking.

### Spatial Coordinate Normalization  
Normalizes all landmarks relative to the wrist position to ensure gesture consistency regardless of hand location on screen.

### Sequential Workflow Engine  
Guides the user through alphabetical ASL collection (`A → Z`) using:
- `P` → Start recording  
- `S` → Stop and move to next letter  
- `Q` → Quit pipeline  

### Automated CSV Compilation  
Stores normalized landmark vectors and labels into:

`asl_hand_data.csv`

with dynamically generated coordinate headers for model training.

---

In [5]:
import cv2
import mediapipe as mp
import csv
import os

# Initialize MediaPipe Hands
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=False, max_num_hands=1, min_detection_confidence=0.7)

# Generate sequence from A to Z
alphabet_list = [chr(i) for i in range(ord('A'), ord('Z') + 1)]
current_letter_idx = 0

cap = cv2.VideoCapture(0) 

data_rows = []
recording = False
samples_collected_for_letter = 0
window_name = "ASL Workflow Manager - CLICK HERE FOR FOCUS"

print("=== Sequential ASL Data Collection Initialized ===")
print("⚠️ CRITICAL: Make sure you CLICK on the video popup window, otherwise keys won't work!")
print(f"Up First: Letter '{alphabet_list[current_letter_idx]}'. Press 'P' to start recording.")

# Create the window explicitly and force it to pop to the front
cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
cv2.setWindowProperty(window_name, cv2.WND_PROP_TOPMOST, 1)

while current_letter_idx < len(alphabet_list):
    ret, frame = cap.read()
    if not ret:
        print("Video stream ended or frame cannot be fetched.")
        break
        
    TARGET_LETTER = alphabet_list[current_letter_idx]
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(rgb)
    
    # Text configs based on status
    if not recording:
        status_text = f"TARGET: '{TARGET_LETTER}' | READY. Press 'P' to Record"
        text_color = (0, 255, 255) # Yellow
    else:
        status_text = f"RECORDING '{TARGET_LETTER}' | Samples: {samples_collected_for_letter} | Press 'S' to Stop"
        text_color = (0, 0, 255) # Red
        
    if results.multi_hand_landmarks:
        for hand_lms in results.multi_hand_landmarks:
            mp.solutions.drawing_utils.draw_landmarks(frame, hand_lms, mp_hands.HAND_CONNECTIONS)
            
            if recording:
                landmarks_row = []
                base_x = hand_lms.landmark[0].x
                base_y = hand_lms.landmark[0].y
                base_z = hand_lms.landmark[0].z
                
                for lm in hand_lms.landmark:
                    landmarks_row.extend([lm.x - base_x, lm.y - base_y, lm.z - base_z])
                
                landmarks_row.append(TARGET_LETTER)
                data_rows.append(landmarks_row)
                samples_collected_for_letter += 1

    # Text backdrop banner
    cv2.rectangle(frame, (10, 10), (630, 65), (20, 20, 20), -1)
    cv2.putText(frame, status_text, (20, 45), cv2.FONT_HERSHEY_SIMPLEX, 0.60, text_color, 2, cv2.LINE_AA)
    
    # Always display helpful instructions at the bottom
    cv2.putText(frame, "Keys -> P: Record | S: Stop/Next | Q: Quit", (20, frame.shape[0] - 20), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
    
    cv2.imshow(window_name, frame)
    
    # Fetch keypress and mask it
    key = cv2.waitKey(1) & 0xFF
    
    # --- Keyboard Logic (Handles both lowercase and uppercase keys now) ---
    if (key == ord('p') or key == ord('P')) and not recording:
        recording = True
        samples_collected_for_letter = 0
        print(f"🔴 Recording started for '{TARGET_LETTER}'...")
        
    elif (key == ord('s') or key == ord('S')) and recording:
        recording = False
        print(f"🟩 Saved '{TARGET_LETTER}' ({samples_collected_for_letter} frames).")
        
        current_letter_idx += 1
        if current_letter_idx < len(alphabet_list):
            print(f"➡️ Up Next: Letter '{alphabet_list[current_letter_idx]}'. Press 'P' when ready.")
        else:
            print("🏁 Reached the end of 'Z'. Compiling your dataset...")
            break
            
    elif key == ord('q') or key == ord('Q'):
        print("Process aborted early by user.")
        break

cap.release()
cv2.destroyAllWindows()

# Save block
if len(data_rows) > 0:
    dataset_file = "asl_hand_data.csv"
    file_exists = os.path.isfile(dataset_file)
    
    print(f"\nWriting dataset records to CSV...")
    with open(dataset_file, "a", newline="") as f:
        writer = csv.writer(f)
        if not file_exists:
            headers = [f"point_{i}_{axis}" for i in range(21) for axis in ['x','y','z']] + ["label"]
            writer.writerow(headers)
        writer.writerows(data_rows)
    print(f"🎉 Success! Data saved cleanly to: {dataset_file}")
else:
    print("⚠️ No data was saved.")

=== Sequential ASL Data Collection Initialized ===
⚠️ CRITICAL: Make sure you CLICK on the video popup window, otherwise keys won't work!
Up First: Letter 'A'. Press 'P' to start recording.
🔴 Recording started for 'A'...
🟩 Saved 'A' (136 frames).
➡️ Up Next: Letter 'B'. Press 'P' when ready.
🔴 Recording started for 'B'...
🟩 Saved 'B' (144 frames).
➡️ Up Next: Letter 'C'. Press 'P' when ready.
🔴 Recording started for 'C'...
🟩 Saved 'C' (123 frames).
➡️ Up Next: Letter 'D'. Press 'P' when ready.
Process aborted early by user.

Writing dataset records to CSV...
🎉 Success! Data saved cleanly to: asl_hand_data.csv


# Random Forest Training Pipeline

## Overview  
This script serves as the machine learning training pipeline for the ASL recognition system.  
It processes the collected landmark dataset (`asl_hand_data.csv`), trains a Random Forest classifier, and exports a deployment-ready model.

---

## Core Modules

### Feature & Label Splitting  
Separates:
- `X` → 63 landmark coordinate features  
- `y` → ASL letter labels (`A–Z`)  

### Stratified Train/Test Split  
Uses an 80/20 split with `stratify=y` to preserve equal class distribution across all letters.

### Random Forest Training  
Builds an ensemble of 100 decision trees using:
- `n_estimators=100`
- `max_depth=20`

to classify hand gestures from landmark structures.

### Model Export  
Evaluates validation accuracy and exports the trained model as:

`asl_rf_model.pkl`

using Joblib serialization for real-time inference deployment.

---

In [18]:
import os
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

dataset_file = "asl_hand_data.csv"
model_export_path = "asl_rf_model.pkl"

if not os.path.exists(dataset_file):
    print(f"❌ Error: Could not find '{dataset_file}'. Please run your data collection script first.")
else:
    print("📖 Loading hand landmark dataset...")
    df = pd.read_csv(dataset_file)
    
    # Split into features (63 joint coordinates) and labels (A-Z)
    X = df.drop("label", axis=1)
    y = df["label"]
    
    # Stratified split to ensure equal letter distribution in validation
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    print("🧠 Training Random Forest Classifier...")
    clf = RandomForestClassifier(n_estimators=100, max_depth=20, random_state=42)
    clf.fit(X_train, y_train)
    
    # Validate accuracy
    accuracy = clf.score(X_test, y_test) * 100
    print(f"✅ Training Complete! Validation Accuracy: {accuracy:.2f}%")
    
    # Save the trained model to disk
    joblib.dump(clf, model_export_path)
    print(f"💾 Model saved successfully as '{model_export_path}'!")

📖 Loading hand landmark dataset...
🧠 Training Random Forest Classifier...
✅ Training Complete! Validation Accuracy: 100.00%
💾 Model saved successfully as 'asl_rf_model.pkl'!


# Live ASL Inference & Word Synthesis Engine

## Overview  
This script acts as the real-time inference and interface engine for the ASL recognition system.  
It captures webcam input, predicts ASL gestures using the trained Random Forest model, and converts stabilized signs into complete words.

---

## Core Modules

### Model Loading  
Loads the trained `asl_rf_model.pkl` classifier and retrieves the expected feature schema for inference.

### Real-Time Landmark Processing  
Uses MediaPipe and OpenCV to extract and normalize 21 hand landmarks into relative coordinate vectors.

### ML Prediction Engine  
Runs `predict_proba()` to generate confidence scores for each ASL letter and filters weak predictions using a confidence threshold.

### Debounce & Word Builder  
Uses a frame-based hold buffer to ensure a sign is stable before appending it to the active word string.

### Dynamic HUD Overlay  
Displays:
- Current predicted letter  
- Confidence score  
- Live synthesized word  
- Animated gesture hold progress bar  

for real-time visual feedback.

---

In [ ]:
import os
import cv2
import joblib
import numpy as np
import pandas as pd  
import mediapipe as mp
import warnings

# 🟢 Suppress the scikit-learn feature name validation warnings completely
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

model_path = "asl_rf_model.pkl"
live_camera_index = 0 

if not os.path.exists(model_path):
    print(f"❌ Error: '{model_path}' not found!")
else:
    clf = joblib.load(model_path)
    print(f"🚀 Loaded trained model from {model_path}")

    # Dynamically extract the exact feature names your model was trained on
    model_features = clf.feature_names_in_ if hasattr(clf, 'feature_names_in_') else None

    mp_hands = mp.solutions.hands
    hands = mp_hands.Hands(
        static_image_mode=False, 
        max_num_hands=1, 
        min_detection_confidence=0.3, 
        min_tracking_confidence=0.3
    )q
    
    cap = cv2.VideoCapture(live_camera_index)
    window_name = "Live ASL Word Generator"
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    if fps <= 0 or fps > 100: fps = 30
        
    current_word = ""
    frame_counter = 0
    last_detected_letter = None
    hold_delay_threshold = int(fps * 0.8) 

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
            
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = hands.process(rgb)
        predicted_letter = None
        max_prob = 0.0
        
        if results.multi_hand_landmarks:
            for hand_lms in results.multi_hand_landmarks:
                mp.solutions.drawing_utils.draw_landmarks(frame, hand_lms, mp_hands.HAND_CONNECTIONS)
                
                live_row = []
                base_x, base_y, base_z = hand_lms.landmark[0].x, hand_lms.landmark[0].y, hand_lms.landmark[0].z
                
                for lm in hand_lms.landmark:
                    live_row.extend([lm.x - base_x, lm.y - base_y, lm.z - base_z])
                
                # Convert the live list into a DataFrame matching training schema
                if model_features is not None:
                    live_df = pd.DataFrame([live_row], columns=model_features)
                else:
                    live_df = pd.DataFrame([live_row])
                
                # Calculate probabilities
                probabilities = clf.predict_proba(live_df)[0]
                best_match_idx = np.argmax(probabilities)
                max_prob = probabilities[best_match_idx]
                best_match_letter = clf.classes_[best_match_idx]
                
                if max_prob > 0.50: 
                    predicted_letter = best_match_letter

        # --- Debounce Logic ---
        if predicted_letter and predicted_letter == last_detected_letter:
            frame_counter += 1
            if frame_counter == hold_delay_threshold:
                current_word += predicted_letter  
                frame_counter = 0 
        else:
            frame_counter = 0
            last_detected_letter = predicted_letter

        # --- UI Overlay ---
        cv2.rectangle(frame, (20, 20), (550, 95), (20, 20, 20), -1)
        cv2.putText(frame, f"Word: {current_word}", (40, 75), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1.3, (50, 255, 50), 3, cv2.LINE_AA)
        
        if predicted_letter:
            cv2.putText(frame, f"Sign: {predicted_letter} ({int(max_prob*100)}%)", (40, 135), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 195, 255), 2, cv2.LINE_AA)
            
            progress_bar_length = int((frame_counter / hold_delay_threshold) * 510)
            cv2.rectangle(frame, (20, 102), (20 + progress_bar_length, 108), (0, 215, 255), -1)
        else:
            cv2.putText(frame, f"Sign: Low Confidence ({int(max_prob*100)}%)", (40, 135), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 0, 255), 2, cv2.LINE_AA)

        cv2.imshow(window_name, frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

🚀 Loaded trained model from asl_rf_model.pkl


KeyboardInterrupt: 

: 

---

## 👨‍💻 About Labellerr's Hands-On Learning in Computer Vision

Thank you for exploring this **Labellerr Hands-On Computer Vision Cookbook**! We hope this notebook helped you learn, prototype, and accelerate your vision projects.  
Labellerr provides ready-to-run Jupyter/Colab notebooks for the latest models and real-world use cases in computer vision, AI agents, and data annotation.

---
## 🧑‍🔬 Check Our Popular Youtube Videos

Whether you're a beginner or a practitioner, our hands-on training videos are perfect for learning custom model building, computer vision techniques, and applied AI:

- [How to Fine-Tune YOLO on Custom Dataset](https://www.youtube.com/watch?v=pBLWOe01QXU)  
  Step-by-step guide to fine-tuning YOLO for real-world use—environment setup, annotation, training, validation, and inference.
- [Build a Real-Time Intrusion Detection System with YOLO](https://www.youtube.com/watch?v=kwQeokYDVcE)  
  Create an AI-powered system to detect intruders in real time using YOLO and computer vision.
- [Finding Athlete Speed Using YOLO](https://www.youtube.com/watch?v=txW0CQe_pw0)  
  Estimate real-time speed of athletes for sports analytics.
- [Object Counting Using AI](https://www.youtube.com/watch?v=smsjBBQcIUQ)  
  Learn dataset curation, annotation, and training for robust object counting AI applications.
---

## 🎦 Popular Labellerr YouTube Videos

Level up your skills and see video walkthroughs of these tools and notebooks on the  
[Labellerr YouTube Channel](https://www.youtube.com/@Labellerr/videos):

- [How I Fixed My Biggest Annotation Nightmare with Labellerr](https://www.youtube.com/watch?v=hlcFdiuz_HI) – Solving complex annotation for ML engineers.
- [Explore Your Dataset with Labellerr's AI](https://www.youtube.com/watch?v=LdbRXYWVyN0) – Auto-tagging, object counting, image descriptions, and dataset exploration.
- [Boost AI Image Annotation 10X with Labellerr's CLIP Mode](https://www.youtube.com/watch?v=pY_o4EvYMz8) – Refine annotations with precision using CLIP mode.
- [Boost Data Annotation Accuracy and Efficiency with Active Learning](https://www.youtube.com/watch?v=lAYu-ewIhTE) – Speed up your annotation workflow using Active Learning.

> 👉 **Subscribe** for Labellerr's deep learning, annotation, and AI tutorials, or watch videos directly alongside notebooks!

---

## 🤝 Stay Connected

- **Website:** [https://www.labellerr.com/](https://www.labellerr.com/)
- **Blog:** [https://www.labellerr.com/blog/](https://www.labellerr.com/blog/)
- **GitHub:** [Labellerr/Hands-On-Learning-in-Computer-Vision](https://github.com/Labellerr/Hands-On-Learning-in-Computer-Vision)
- **LinkedIn:** [Labellerr](https://in.linkedin.com/company/labellerr)
- **Twitter/X:** [@Labellerr1](https://x.com/Labellerr1)

*Happy learning and building with Labellerr!*
